In [ ]:
!pip install news-please
!pip install -U easynmt
!pip install nltk
import pandas as pd
import numpy as np
import requests
import json
import re
from bs4 import BeautifulSoup
from newsplease import NewsPlease
from easynmt import EasyNMT
from torch import cuda
import nltk
nltk.download('punkt_tab')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.5/960.5 kB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# **Scraping BiH Presidency for Speech Links**
Using the website index 1-44 to scrape links to all of the pages with speeches

In [ ]:
from os import wait3
def getPages():
  index = 1
  pres_site = 'https://www.predsjednistvobih.ba/gov/Archive.aspx?langTag=bs-BA&template_id=156&pageIndex='
  speech_ids = []

  while index < 68:
    page = 'https://www.predsjednistvobih.ba/gov/Archive.aspx?langTag=bs-BA&template_id=156&pageIndex=' + str(index)

    r = requests.get(page)
    soup = BeautifulSoup(r.text, "html.parser")
    tags = soup.find_all('a', href=re.compile(r'/gov/\?id=\d+'))
    for tag in tags:
      id = re.search(r'[0-9][0-9][0-9][0-9][0-9][0-9]', str(tag))
      if id: # Check if id is not None
        speech_ids.append(id.group())
      else:
        id = re.search(r'[0-9][0-9][0-9][0-9][0-9]', str(tag))
        if id: # Check if id is not None
          speech_ids.append(id.group())
    index += 1
  return speech_ids

In [ ]:
def getSpeech(speech_ids):
  titles = []
  speech_urls = []
  publish_date = []
  content = []
  for id in speech_ids:
    speech_url = 'https://www.predsjednistvobih.ba/gov/default.aspx?id=' + id + '&langTag=bs-BA'
    speech = NewsPlease.from_url(speech_url)
    titles.append(speech.title)
    speech_urls.append(speech_url)
    publish_date.append(speech.date_publish)
    content.append(speech.maintext)
  return pd.DataFrame({
      'id': speech_ids,
      'title': titles,
      'speech_url': speech_urls,
      'publish_date': publish_date,
      'content': content
  })


In [ ]:
speech_ids = getPages()
df = getSpeech(speech_ids)
df.to_csv('speech_data_BS.csv', index=False)

In [ ]:
df = pd.read_csv('/content/speech_data_BS.csv')

In [ ]:
def TranslateSpeeches(df):
  entries = df['content']
  model = EasyNMT('opus-mt')
  opus_mt = []
  df_translation = df.copy()
  for t in model.translate_stream(entries, source_lang = "sla", target_lang = "en", show_progress_bar = True, batch_size=5):
      print(t)
      print("___________")
      opus_mt.append(t)

  df_translation['opus_mt'] = opus_mt
  return df_translation


In [ ]:
df_translation = TranslateSpeeches(df)
df_translation.to_csv('speech_data_BS_translated.csv', index=False)

11.9kB [00:00, 20.7MB/s]                   
/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
  0%|          | 0/665 [00:00<?, ?it/s][nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (h

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/860k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/299M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]